<a href="https://colab.research.google.com/github/LikithaSrinivasa25/ml-dl-algorithms/blob/main/machine-learning/Data_Pre_processing_logistic_regressio.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

___
# COMP1804 Week 3 - Data Pre-processing and logistic regression



**Learning Objectives:**
 *  Understand the stages of data pre-processing.
 *  Practice the different stages of data pre-processing in Python.
 *  Use logistic regression for a classification task.

I will show you how to apply various pre-processing steps using the training dataset as an example. Generally, the exercises today will be about applying the same steps to the test dataset. Some times, you will have to apply slightly modified steps to both the training and the test dataset.



> **HINT**: always start by copying the content of the last code cell before the exercise. Then, think about which modification you need to apply. Sometime, it's about changing a variable, sometime it's about changing an argument inside a python function.


___


Credits:
Tutorial adapted from previous ones, originally delivered by Dimitris Kollias and modified by Stef Garasto.

## 1. Loading the Dataset


For this Lab, I have used a subset of the Loan Prediction dataset. You should have downloaded the training dataset together with this notebook. If you haven't, go back to Moodle.


Now, lets get started by importing important packages and the dataset.

**1.1 Import the necessary Python modules**

In [ ]:
# Load python modules

import numpy as np # numpy is a library that allows us to work with vectors and matrices
import matplotlib.pyplot as plt # visualisation library
import pandas as pd # pandas is a library that allows us to work with DataFrames

# On some level, dataframes are enhanced matrices where we have assigned names to each
# row and each column.


1.2 Load Dataset

First, download the csv files from Moodle to your local drive and unzip the folder.

Then, if using Google Colab, upload the two .csv dataset to the home folder of your Google drive. You can then run the cell below to connect Google Colab to your Google Drive. It will ask you for permission, but it is only a standard request.

You can skip the next step if working on your local machine.

In [ ]:
# Connect google colab to your google drive.
# Note that you need to be logged in to your google account for this step to work
USING_COLAB = True # change this to false if you're working on a local machine

if USING_COLAB:
  from google.colab import drive
  drive.mount('/content/drive')


Mounted at /content/drive


You should now see your drive folder into the file explorer tab, on the left (to access it, click on the folder-like icon, highlighted in orange in the figure below).

If you click on the arrow next to "drive" (see figure below), it will show you the content of your drive. To load into python any file from your google drive, remember to start the file path with `/content/drive'. To easily get the full path of a file in your google drive, you can navigate to it in the file explorer tab, click on the three dots next to it, and click on "copy path" (see image below).



<div>
<img src="https://drive.google.com/uc?export=view&id=1exU5qHVmvzbbMhdkyQzXttBtEHpMth1X" width="500"/>
</div>


In [ ]:
# NOTE! If you are working on a local machine, replace the path below with the path
# to where the (unzipped) file was saved on your local machine.
X = pd.read_csv('/content/X_loan.csv')
Y = pd.read_csv('/content/Y_loan.csv')


First, an observation. This dataset has labels, as made clear by the fact that there is a dedicated dataset storing them (typically, the variable Y indicates labels, or outputs, and the label X indicates input features). Since we have labels, we can use them to train our model: this is a **supervised** ML task. The X dataset stores the **input** to the model, the Y dataset represents the desired **output** of the model.

At times, you will have the input and output variables in the same dataset. We will see examples of this in future tutorials.


## Dataset Split

Let's split both the input features and the labels into a training set and a test set. Scikit-learn has a helpful function for this: "train_test_split".

The function can split multiple datasets at the same time. Note that if the datasets you need to split are related (that is, each row from one dataset correspond to the same row from another dataset), you **need** to split them at the same time. Otherwise, the correspondence will get lost (remember the exercise from week 2).


Although, there is one EDA check that we can perform before dataset splitting. This is: checking for duplicates and dropping any that we find. This was we avoid identical copies of a data point to end up in both the training and test dataset.

We can use the pandas function `.duplicated()` to get which rows are duplicates, so that we can drop them from both the input and the output datasets.

In [ ]:
duplicates_rows = X.duplicated()
# The result is True for rows that are a duplicate, False for rows that are not a duplicate

print(f'There are {duplicates_rows.sum()} duplicates rows.')

# We have no duplicates here,
# but if we had them, to remove them we'd only keep the rows that are NOT duplicates
X = X[~duplicates_rows]
Y = Y[~duplicates_rows]


There are 0 duplicates rows.


In [ ]:
from sklearn.model_selection import train_test_split

# remember we do a "stratified" split, so that we can make sure there is the
# same proportion of labels in both the training and the test dataset.
# This is particularly helpful if the dataset is imbalanced.
# scikit-learn should have "shuffle=True" by default, but best to specify it
# so that we are sure the split between training and test is really random
train_dataset, test_dataset, train_labels, test_labels = train_test_split(X, Y,
                                                                          test_size= 0.2,
                                                                          stratify = Y,
                                                                          shuffle=True)


## 2. Data quality assessment and Exploratory Data Analysis

It is good practice to spend some time exploring the data to find out any issues as early as possible.

There can be many data quality issues, including invalid/inconsistent features or data points, unnecessary features, missing values, and more.

First, let's inspect the dataset a bit.

#### Dimensions of Dataset and feature names


In [ ]:
# Training data
# shape of input:
# The number or rows is the number of data points
# The number or columns is the number of features
print(train_dataset.shape)
# shape of output:
# The number or rows is the number of data points (should be the same as before!)
# The number or columns is the number of labels we want to predict
print(train_labels.shape)

(312, 12)
(312, 1)


In [ ]:
# Test data
# shape of input:
# The number or rows is the number of data points
# The number or columns is the number of features (should be the same as for the training!)
print(test_dataset.shape)
# shape of output:
# The number or rows is the number of data points (should be the same as before!)
# The number or columns is the number of labels we want to predict
print(test_labels.shape)

(79, 12)
(79, 1)


In [ ]:
# list of column titles
print(train_dataset.columns)
print(train_labels.columns)

Index(['Loan_ID', 'Gender', 'Married', 'Dependents', 'Education',
       'Self_Employed', 'ApplicantIncome', 'CoapplicantIncome', 'LoanAmount',
       'Loan_Amount_Term', 'Credit_History', 'Property_Area'],
      dtype='object')
Index(['Target'], dtype='object')


In [ ]:
# list of column (field) data types. doing this on the test datasets will give the same results
print(train_dataset.dtypes)
print(train_labels.dtypes)

# Note: object is a Pandas data type for pretty much anything that is not a number


Loan_ID               object
Gender                object
Married               object
Dependents            object
Education             object
Self_Employed         object
ApplicantIncome      float64
CoapplicantIncome    float64
LoanAmount           float64
Loan_Amount_Term     float64
Credit_History       float64
Property_Area         object
dtype: object
Target    object
dtype: object


### Take a peek at the Dataset

Note that if you see "NaN", it means "Not a Number". It is not the same as 0. Python replaces empty/missing fields in the data with "NaN".


In [ ]:
# you can show the first N rows in a dataframe with the function "head"
print(train_dataset.head(10))
# note how the first row has a missing value! We'll get back to it.
# Also notice how the first two rows are almost identical, aside from the Loan_ID and the missing value
# Could this be the same application submitted twice because the first time someone forgot to add the income?

      Loan_ID  Gender Married Dependents     Education Self_Employed  \
77   LP002342    Male     Yes          2      Graduate           Yes   
26   LP001713    Male     Yes          1      Graduate           Yes   
41   LP001194    Male     Yes          2      Graduate            No   
203  LP001245    Male     Yes          2  Not Graduate           Yes   
278  LP001917  Female      No          0      Graduate            No   
215  LP002459    Male     Yes          0      Graduate            No   
148  LP001915    Male     Yes          2           NaN            No   
366  LP001570    Male     Yes          2      Graduate            No   
123  LP002893   other      No          0      Graduate            No   
371  LP002974    Male     Yes          0      Graduate            no   

     ApplicantIncome  CoapplicantIncome  LoanAmount  Loan_Amount_Term  \
77            1600.0            20000.0       239.0             360.0   
26            7787.0                0.0       240.0          

In [ ]:
# you can also show a random subset of the data with the function "sample".
# Sometimes it's better to show a sample, so that you get a more varied view of the data
train_dataset.sample(10)

,Loan_ID,Gender,Married,Dependents,Education,Self_Employed,ApplicantIncome,CoapplicantIncome,LoanAmount,Loan_Amount_Term,Credit_History,Property_Area
206,LP001100,Male,No,3+,Graduate,No,12500.0,3000.0,320.0,360.0,1.0,Rural
198,LP002743,Female,No,0,Graduate,No,2138.0,0.0,99.0,360.0,0.0,Semiurban
202,LP001349,Male,No,0,Graduate,No,4843.0,3806.0,151.0,360.0,1.0,Semiurban
30,LP001891,Male,Yes,0,Graduate,No,11146.0,0.0,136.0,360.0,1.0,Urban
4,LP001814,Male,Yes,2,Graduate,No,9703.0,0.0,112.0,360.0,1.0,Urban
123,LP002893,other,No,0,Graduate,No,1836.0,33837.0,90.0,360.0,1.0,Urban
19,LP001491,Male,Yes,2,Graduate,Yes,3316.0,3500.0,88.0,360.0,1.0,Urban
89,LP002403,Male,No,0,Graduate,Yes,10416.0,0.0,187.0,360.0,0.0,Urban
120,LP002586,Female,Yes,1,Graduate,No,3326.0,913.0,105.0,84.0,1.0,Semiurban
146,LP001940,Male,Yes,2,Graduate,No,3153.0,1560.0,134.0,360.0,1.0,Urban


For large dataset it is impractical to go through each data point one by one (although it is not unheard of for small to medium datasets). So, we need to define some rules for checking. We can do this more easily if we know (or can guess) what to expect from the data, but this might not always be possible. Exploratory data analysis often goes together with data quality assessment. After all, we need to explore the data to spot any issues!


Next EDA steps: use Pandas `describe()` function to get some high-level statistics about the data

For NUMERICAL features, we can get the following info:

*   count = Count number of non-NA/null observations for a feature (see also below).
*   mean = average value of each feature.
*   std = standard deviation of the values in each feature
*   min = Minium of the values in the feature.
*   max = Maximum of the values in the feature.
*   25%, 50%, 75% = 25%, 50%, 75% percentiles in each feature. 50% percentile=median

Here is a good explanation on what percentiles are: https://simple.wikipedia.org/wiki/Percentile.

In [ ]:
# Summary statistics for numerical features
train_dataset.describe()


,ApplicantIncome,CoapplicantIncome,LoanAmount,Loan_Amount_Term,Credit_History
count,309.000000,309.000000,312.000000,311.000000,311.000000
mean,5211.333333,1555.505243,140.426282,343.755627,0.858521
std,5701.230455,2759.058926,76.287119,62.699628,0.349077
min,645.000000,0.000000,-21.000000,60.000000,0.000000
25%,2920.000000,0.000000,100.000000,360.000000,1.000000
50%,3887.000000,1032.000000,125.000000,360.000000,1.000000
75%,5818.000000,2253.000000,161.250000,360.000000,1.000000
max,81000.000000,33837.000000,600.000000,480.000000,1.000000


For CATEGORICAL features, we want to use the Pandas function `value_counts()` to obtain information about how many categories there are and how many instances of each.

In [ ]:
# let's list all categorical features
categorical_columns= ['Gender', 'Married', 'Dependents', 'Education',
       'Self_Employed', 'Property_Area']
# Question: why have we not included Loan_ID?

# let's get the categories and their count for each feature
for col in categorical_columns:
  print(f"Categories and number of occurrences for '{col}'")
  print(train_dataset[col].value_counts())
  print()

Categories and number of occurrences for 'Gender'
Gender
Male      253
Female     50
other       8
Name: count, dtype: int64

Categories and number of occurrences for 'Married'
Married
Yes    199
No     112
Name: count, dtype: int64

Categories and number of occurrences for 'Dependents'
Dependents
0     187
2      54
1      48
3+     22
Name: count, dtype: int64

Categories and number of occurrences for 'Education'
Education
Graduate        245
Not Graduate     66
Name: count, dtype: int64

Categories and number of occurrences for 'Self_Employed'
Self_Employed
No     267
Yes     42
no       3
Name: count, dtype: int64

Categories and number of occurrences for 'Property_Area'
Property_Area
Semiurban    125
Urban         98
Rural         86
London         2
Name: count, dtype: int64



Wait! Some things don't make much sense:


1.   Why is there a category "London" in "Property_Area"? This feature describes the type of area someone lives in, not the city.
2.   Why is the minimum "Loan Amount" a negative value?
3.   Are "no" and "No" the same category for the "Self_Employed" feature? We can guess that yes, they do mean the same thing.


These are all mistakes that may have arisen from human or machine errors. How do we fix them?

Well, negative income and a wrong value for a given category really means that we do not have information about that specific value. In practice, it's as if it were a missing value, we might as well replace it with on. The mismatched categories can be merged, instead.

Two notes:
1. In some cases, the wrong values still give us relevant information. For example, if we could be sure that "London" is actually the city where the applicant lives, we could reasonably assume that the corresponding "Property_Area" value would be "Urban". However, these decisions are highly case-specific.
2. The "Other" in the gender category is not a mistake. Many data collection processes do include options for genders that are different from "Male" and "Female".

### Data cleaning

In [ ]:
# replace 'yes' with 'Yes' for 'self_employed'
# remember that .loc is a way to access a subset of the dataframe
# We can write df.loc[condition_A, condition_B] where condition_A determines
# which rows we select and condition_B determines which columns we select
train_dataset.loc[train_dataset.Self_Employed=='no','Self_Employed'] = 'No'

# replace negative loan amount values with NaN
# (we use np.nan because that's how the other missing values are represented in this dataset)
train_dataset.loc[train_dataset.LoanAmount<0,'LoanAmount'] = np.nan

# replace out of scope Property_Area values with NaN
accepted_property_areas = ['Urban','Rural','Semiurban']
# remember the lambda notation: it is an anonymous function that lets us specificy
# a given (and easy) transformation for its input (in this case, x)
train_dataset.loc[train_dataset.Property_Area.map(lambda x: x not in accepted_property_areas),'Property_Area'] = np.nan


### Exercise 1
Repeat all the steps above for the test dataset. Note that we have only used the training dataset to decide how to clean the data (to avoid data leakage).

Remember: you can start by copying the content of the last code cell above. Then, think about what you need to change to apply it to the test dataset instead (what's the right variable name?).

In [ ]:
# YOUR CODE HERE
# list of column titles
print(test_dataset.columns)
print(test_labels.columns)

Index(['Loan_ID', 'Gender', 'Married', 'Dependents', 'Education',
       'Self_Employed', 'ApplicantIncome', 'CoapplicantIncome', 'LoanAmount',
       'Loan_Amount_Term', 'Credit_History', 'Property_Area'],
      dtype='object')
Index(['Target'], dtype='object')


In [ ]:
print(test_dataset.head(10))

      Loan_ID Gender Married Dependents     Education Self_Employed  \
12   LP002149   Male     Yes          2      Graduate            No   
236  LP002767   Male     Yes          0      Graduate            No   
48   LP002689   Male      No          0      Graduate            No   
337  LP002529   Male     Yes          2      Graduate            No   
133  LP002119   Male     Yes          1  Not Graduate            No   
13   LP002472   Male      No          2      Graduate            No   
14   LP001647   Male     Yes          0      Graduate            No   
149  LP001915   Male     Yes          2      Graduate            No   
341  LP002961   Male     Yes          1      Graduate            No   
299  LP002555   Male     Yes          2      Graduate           Yes   

     ApplicantIncome  CoapplicantIncome  LoanAmount  Loan_Amount_Term  \
12            8333.0        3167.000000       165.0             360.0   
236           2768.0        1950.000000       155.0             360.0   

In [ ]:
# Summary statistics for numerical features
test_dataset.describe()


,ApplicantIncome,CoapplicantIncome,LoanAmount,Loan_Amount_Term,Credit_History
count,78.000000,79.000000,78.000000,78.000000,79.000000
mean,4827.871795,1672.048101,141.243590,334.307692,0.848101
std,3169.301723,2015.676282,74.288169,77.617830,0.361216
min,150.000000,0.000000,17.000000,36.000000,0.000000
25%,2772.250000,0.000000,100.000000,360.000000,1.000000
50%,3909.000000,1425.000000,130.000000,360.000000,1.000000
75%,5861.500000,2319.500000,167.250000,360.000000,1.000000
max,16120.000000,11300.000000,495.000000,480.000000,1.000000


In [ ]:
# let's list all categorical features
categorical_columns= ['Gender', 'Married', 'Dependents', 'Education',
       'Self_Employed', 'Property_Area']
# Question: why have we not included Loan_ID?

# let's get the categories and their count for each feature
for col in categorical_columns:
  print(f"Categories and number of occurrences for '{col}'")
  print(test_dataset[col].value_counts())
  print()

Categories and number of occurrences for 'Gender'
Gender
Male      63
Female    13
other      3
Name: count, dtype: int64

Categories and number of occurrences for 'Married'
Married
Yes    55
No     24
Name: count, dtype: int64

Categories and number of occurrences for 'Dependents'
Dependents
0     43
2     16
1     13
3+     7
Name: count, dtype: int64

Categories and number of occurrences for 'Education'
Education
Graduate        67
Not Graduate    12
Name: count, dtype: int64

Categories and number of occurrences for 'Self_Employed'
Self_Employed
No     66
Yes    12
no      1
Name: count, dtype: int64

Categories and number of occurrences for 'Property_Area'
Property_Area
Rural        27
Urban        27
Semiurban    24
London        1
Name: count, dtype: int64



In [ ]:
# replace 'yes' with 'Yes' for 'self_employed'
# remember that .loc is a way to access a subset of the dataframe
# We can write df.loc[condition_A, condition_B] where condition_A determines
# which rows we select and condition_B determines which columns we select
test_dataset.loc[test_dataset.Self_Employed=='no','Self_Employed'] = 'No'

# replace negative loan amount values with NaN
# (we use np.nan because that's how the other missing values are represented in this dataset)
test_dataset.loc[test_dataset.LoanAmount<0,'LoanAmount'] = np.nan

# replace out of scope Property_Area values with NaN
accepted_property_areas = ['Urban','Rural','Semiurban']
# remember the lambda notation: it is an anonymous function that lets us specificy
# a given (and easy) transformation for its input (in this case, x)
test_dataset.loc[test_dataset.Property_Area.map(lambda x: x not in accepted_property_areas),'Property_Area'] = np.nan


## Managing Missing Data

Sometimes you may find some data are missing in the dataset. If the missing values are not handled properly inaccurate inference about the data may result. Due to improper handling, the result obtained will differ from ones where the missing values are present.

Since missing values can tangibly reduce prediction accuracy, this step needs to be a priority. In terms of machine learning, assumed or approximated values are “more appropriate” for an algorithm than just missing ones.  Even if you don’t know the exact value, methods exist to better “assume” which value is missing or bypass the issue. However, keep in mind that this might also be introducing biases in your dataset, especially when data is not missing at random (For example, some people might be more reluctant than other to disclose their, let's say, immigration status, for multiple reasons. In this case, an "assumed" value might not work as well).

First, though, let's find out how many missing values (or NaN values) there are in each feature, using Pandas `isna()` function.

In [ ]:
# Number of missing values per column
# isna(). returns a True/False value for each element in the dataframe/
# True if that value is a NaN value, False otherwise
# calling .sum() sums the number of True values in each column
# So, the output is the number of missing values in each column
print(train_dataset.isna().sum())


Loan_ID              0
Gender               1
Married              1
Dependents           1
Education            1
Self_Employed        0
ApplicantIncome      3
CoapplicantIncome    3
LoanAmount           2
Loan_Amount_Term     1
Credit_History       1
Property_Area        3
dtype: int64


In [ ]:
# another function you can use is df.info()
# df.info() which gives you the number of NON-null datapoints (so, the higher the better)
# If the amount of non-null for any column is less than the total number of data
# points then that feature has missing values.
# For example, below notice how "ApplicantIncome" only has 309 non-null out of 312.
# df.info() also tells you what data type a feature is, like the .dtypes call above
print(train_dataset.info())

<class 'pandas.core.frame.DataFrame'>
Index: 312 entries, 77 to 180
Data columns (total 12 columns):
 #   Column             Non-Null Count  Dtype  
---  ------             --------------  -----  
 0   Loan_ID            312 non-null    object 
 1   Gender             311 non-null    object 
 2   Married            311 non-null    object 
 3   Dependents         311 non-null    object 
 4   Education          311 non-null    object 
 5   Self_Employed      312 non-null    object 
 6   ApplicantIncome    309 non-null    float64
 7   CoapplicantIncome  309 non-null    float64
 8   LoanAmount         310 non-null    float64
 9   Loan_Amount_Term   311 non-null    float64
 10  Credit_History     311 non-null    float64
 11  Property_Area      309 non-null    object 
dtypes: float64(5), object(7)
memory usage: 31.7+ KB
None


### Removing Missing Data

So how can we handle missing data? One obvious idea is to remove the lines (observations recorded in the rows) where there is some missing data. That could be ok in some circumstances, but then we wouldn't be able to handle any future input with missing data. Also, the dataset we have here is considerably small, so removing data will delete relevant information and can have a crucial impact. So, we might want to look for alternative methods to deal with the missing data.

When the missing values are from the label columns, then it can be appropriate to remove those rows. This is because if we don't know what the "correct" label is for a data point, we are kind of missing the relevant information for a supervised ML algorithm to learn.

Assuming it was appropriate to remove observation rows, the following code examples will help. They use the pandas function `dropna()` and `drop()`, in various ways.



**3.1.1 If we wanted to remove all rows that contain missing data**


In [ ]:
# remove all rows with missing data
# dropna removes all rows that contain at least one missing value
print(f'Original dataset length: {len(train_dataset)}')
print(f'Dataset length after removing missing rows: {len(train_dataset.dropna())}')



Original dataset length: 312
Dataset length after removing missing rows: 303


**3.1.2 If we wanted to remove specific rows**

If we already know which rows we want to drop, we can give this information to the `dropna()` function. Specifically, we can pass the index values of the undesired rows to the drop method.

In [ ]:
# remove selected rows (e.g. the third row, which has index `2`)
# Notice the difference in the output from before to after! Look at the Load_ID column
row_index= train_dataset.index[1]
print('Before dropping a row')
print(train_dataset.head(3))

print()
print('After dropping a row (notice the second row is not there anymore)')
print(train_dataset.drop([row_index]).head(3))



Before dropping a row
     Loan_ID Gender Married Dependents Education Self_Employed  \
77  LP002342   Male     Yes          2  Graduate           Yes   
26  LP001713   Male     Yes          1  Graduate           Yes   
41  LP001194   Male     Yes          2  Graduate            No   

    ApplicantIncome  CoapplicantIncome  LoanAmount  Loan_Amount_Term  \
77           1600.0            20000.0       239.0             360.0   
26           7787.0                0.0       240.0             360.0   
41           2708.0             1167.0        97.0             360.0   

    Credit_History Property_Area  
77             1.0         Urban  
26             1.0         Urban  
41             1.0     Semiurban  

After dropping a row (notice the second row is not there anymore)
      Loan_ID Gender Married Dependents     Education Self_Employed  \
77   LP002342   Male     Yes          2      Graduate           Yes   
41   LP001194   Male     Yes          2      Graduate            No   
203 

**3.1.3 If we wanted to remove specific columns**

Usually you would drop particular columns especially if all or most of its values are missing. The drop method can also be used here with parameters to define the column title and axis=1 to denote that we want to drop a column.


In [ ]:
# remove selected column - notice how the output dataframe doesn't have the "Gender" column anymore
# axis = 1 tells Pandas that the keyword "Gender" is the name of a column, not a row
print(train_dataset.drop("Gender", axis=1).head(2))


     Loan_ID Married Dependents Education Self_Employed  ApplicantIncome  \
77  LP002342     Yes          2  Graduate           Yes           1600.0   
26  LP001713     Yes          1  Graduate           Yes           7787.0   

    CoapplicantIncome  LoanAmount  Loan_Amount_Term  Credit_History  \
77            20000.0       239.0             360.0             1.0   
26                0.0       240.0             360.0             1.0   

   Property_Area  
77         Urban  
26         Urban  


In [ ]:
# remove multiple selected columns
# notice how now both columns "Gender" and "Married" are missing

print(train_dataset.drop(["Gender", "Married"], axis=1).head(2))


     Loan_ID Dependents Education Self_Employed  ApplicantIncome  \
77  LP002342          2  Graduate           Yes           1600.0   
26  LP001713          1  Graduate           Yes           7787.0   

    CoapplicantIncome  LoanAmount  Loan_Amount_Term  Credit_History  \
77            20000.0       239.0             360.0             1.0   
26                0.0       240.0             360.0             1.0   

   Property_Area  
77         Urban  
26         Urban  


**3.1.4 If we wanted to remove all rows where data satisfies a condition in a particular column**

Drop all rows where Married is "No" (or a NaN value). Since there are two categories in the Married column, that is equivalent to filtering the dataframe where Married = Yes.


In [ ]:
# remove all rows where Married is No or NaN.
# This condition is equivalent to keeping all the rows where Married is Yes
# So, we can filter the dataframe by only keeping those rows that satisfy the condition above
# Notice how the Married column only has "Yes" now
print((train_dataset[train_dataset["Married"] == "Yes"]).head(5))

      Loan_ID Gender Married Dependents     Education Self_Employed  \
77   LP002342   Male     Yes          2      Graduate           Yes   
26   LP001713   Male     Yes          1      Graduate           Yes   
41   LP001194   Male     Yes          2      Graduate            No   
203  LP001245   Male     Yes          2  Not Graduate           Yes   
215  LP002459   Male     Yes          0      Graduate            No   

     ApplicantIncome  CoapplicantIncome  LoanAmount  Loan_Amount_Term  \
77            1600.0            20000.0       239.0             360.0   
26            7787.0                0.0       240.0             360.0   
41            2708.0             1167.0        97.0             360.0   
203           1875.0             1875.0        97.0             360.0   
215           4301.0                0.0       118.0             360.0   

     Credit_History Property_Area  
77              1.0         Urban  
26              1.0         Urban  
41              1.0     Se

**3.1.5 If we wanted to remove all rows where data is missing in a particular column**

Remove all rows where a particular column has a missing value. This will result in a dataset that has no missing values in that particular column. It may still contain missing values in other columns.


In [ ]:
# remove all rows where ApplicantIncome is missing.
# 1. check to see if there are missing data in the ApplicantIncome column
print('Number of missing ApplicantIncome data: ')
print(train_dataset["ApplicantIncome"].isnull().sum())

print()

# 2. drop all rows where a value for ApplicantIncome is missing
# we can do this by specifying the column of interest as a parameter to the dropna function
# Here, pandas only drop rows if they have a missing value in the column "ApplicantIncome"
# It will ignore any other missing value
print(train_dataset.dropna(subset=["ApplicantIncome"]))
print('-'*30)
# the first row contains a null and is removed

# alternative way:
print('\n Notice how missing values are still there, but not in the ApplicantIncome column: \n')
train_dataset.dropna(subset=["ApplicantIncome"]).isna().sum()


Number of missing ApplicantIncome data: 
3

      Loan_ID  Gender Married Dependents     Education Self_Employed  \
77   LP002342    Male     Yes          2      Graduate           Yes   
26   LP001713    Male     Yes          1      Graduate           Yes   
41   LP001194    Male     Yes          2      Graduate            No   
203  LP001245    Male     Yes          2  Not Graduate           Yes   
278  LP001917  Female      No          0      Graduate            No   
..        ...     ...     ...        ...           ...           ...   
355  LP002984    Male     Yes          2      Graduate            No   
161  LP002837    Male     Yes         3+      Graduate            No   
361  LP001073    Male     Yes          2  Not Graduate            No   
281  LP002978  Female      No          0      Graduate            No   
180  LP001954  Female     Yes          1      Graduate            No   

     ApplicantIncome  CoapplicantIncome  LoanAmount  Loan_Amount_Term  \
77            1600

,0
Loan_ID,0
Gender,0
Married,1
Dependents,0
Education,1
Self_Employed,0
ApplicantIncome,0
CoapplicantIncome,1
LoanAmount,2
Loan_Amount_Term,0


### Filling in Missing Data

Dropping rows or columns might work in some cases. Other times, though, this way of proceeding would mean dropping too much data. Choosing the right approach also heavily depends on data and the domain you have:


As alternative to removing rows with missing data, you can estimate plausible values for the missing data instead. For example, you can:
* Substitute missing values with "fake" values, e.g. "n/a" for categorical or 0 for numerical values. This will retain the information that a data value is missing, which might be important in some cases, but it doesn't really solve the problem.
* For numerical features, substitute the missing numerical values with the mean or the median value of all the non-missing elements in the column.
* For categorical features, substitute the missing values with the most frequent item in a column.

The latter two are called imputation methods. To apply them, we will use the scikit-learn library. The impute class allows us to manipulate the missing data.

At first we will create an object of the imputer class.
The class has an argument called 'strategy' which shows what method to use to generate a value replacement for the missing value. 'mean' is the default value for strategy.

If “mean”, then replace missing values using the mean along each column. Can only be used with numeric data.

If “median”, then replace missing values using the median along each column. Can only be used with numeric data.

If “most_frequent”, then replace missing using the most frequent value along each column. Can be used with strings or numeric data. If there is more than one such value, only the smallest is returned.

If “constant”, then replace missing values with fill_value. Can be used with strings or numeric data.

*********************
#### IMPORTANT NOTE!

You may have noticed a flaw in the plan. Let's say we want to replace missing numerical values with the average value in the column. First, we need to know what the average value is!

This is also an example of a "parameter fitting" procedure. Parameters fitting procedures are used to calculate the values of important parameters that are used again and again in a machine learning application (like the parameters for the ML models themselves).

The mean value of a numerical feature is one of these parameters when we want to handle missing values of a numerical feature.

To put this into practice, there is a typical workflow with sklearn functions. These functions are typically objects which follow a specific pipeline:
1. create an instance of a given object (like the imputer class)
2. fit this instance on the "known" dataset only, that is the training dataset. This is done via the attribute `.fit()`. Remember that if we used the test dataset too, doing so could lead to data leakage.
3. use the instance on all the datasets (for now, this means both training and testing). This is done via the attribute `.transform()`.

Note that sklearn also allows you to fit and transform all at once by calling the attribute `.fit_transform()`.

Remember this pipeline. We have already seen a variation of it in week 2 with the `.fit()` then `.predict()` sequence and it will appear again and again.

<br/><br/>

<div>
<img src="https://drive.google.com/uc?export=view&id=1p-R8TJ6xCBaEw4XKDD6F1dCTJp5Vcb8z" width="500"/>
</div>


**********************

In [ ]:
# number of missing values for Gender and ApplicantIncome
print(train_dataset["Gender"].isnull().sum())

print(train_dataset["ApplicantIncome"].isnull().sum())



1
3


In [ ]:
# handling missing data
from sklearn.impute import SimpleImputer

train_dataset_no_nans =  train_dataset.copy()

# 1. Imputer
imptr_num = SimpleImputer(missing_values = np.nan, strategy = 'mean')


# 2. Fit the imputer object to the feature matrix (only for numeric features)
numerical_columns = ['ApplicantIncome', 'CoapplicantIncome',
                'LoanAmount', 'Loan_Amount_Term', 'Credit_History']
imptr_num = imptr_num.fit(train_dataset_no_nans[numerical_columns]) # fit the data to estimate the parameters (here, the average value)

# 3. Call Transform to replace missing data in train_dataset (on specific columns) by the mean of the column to which that missing data belongs to
train_dataset_no_nans[numerical_columns] = \
  imptr_num.transform(train_dataset_no_nans[numerical_columns]) # apply the transformation using the parameters estimated above

# Let's check we have fewer missing values than before
# (check the column ApplicantIncome for example and compare with before)
print(train_dataset_no_nans.isnull().sum())



Loan_ID              0
Gender               1
Married              1
Dependents           1
Education            1
Self_Employed        0
ApplicantIncome      0
CoapplicantIncome    0
LoanAmount           0
Loan_Amount_Term     0
Credit_History       0
Property_Area        3
dtype: int64


Now for an example on the categorical feature. We will start by applying the imputer to only one category.

This time we will use the "most_frequent" strategy.
Most people are not self employed, but is it safe to assume that we can replace all missing values with "not self-employed"?

In [ ]:
# 1. Imputer
imptr_example = SimpleImputer(missing_values = np.nan, strategy = 'most_frequent')

# 2. Fit the imputer object to the feature matrix
imptr_example = imptr_example.fit(train_dataset_no_nans[['Self_Employed']])

# 3. Call Transform to replace missing data in train_dataset (on specific columns) by the mean of the column to which that missing data belongs to
train_dataset_no_nans[['Self_Employed']] = imptr_example.transform(
    train_dataset[['Self_Employed']])

# check the column has no more missing values:
print(train_dataset['Self_Employed'].isna().sum())



0


##############################################################################





###  Exercise 2

Fill in the missing values for all the categorical features together.
Remember that we made a list of categorical features above and saved it into the variable `categorical_columns`.

Call the SimpleImputer variable: `imptr_cat`.

Work on the variable: `train_dataset_no_nans`.

Remember that the steps are: 1. instantiate the imputer, use `.fit()` with the training dataset, then use `.transform()` and save the results in the same dataset.









In [ ]:
### insert code here; work on the: train_dataset_no_nans



##############################################################################




## 4. Encoding categorical Data

Data Preprocessing in machine learning requires values of the data in numerical form. Therefore text values in the columns of datasets must be converted into numerical form.

### Converting categorical to numerical values

Given the original dataset, it is clear we have multiple categorical features. All these need to be encoded. The [LabelEncoder](https://scikit-learn.org/stable/modules/generated/sklearn.preprocessing.LabelEncoder.html) class is used to transform the categorical or string values to numerical ones (between 0 and n_classes-1).



In [ ]:
# note we are using the dataset with no missing values
temp_dataset = train_dataset_no_nans.copy() #since this is not the final transformation, we make a copy and use a temporary variable name

# encode categorical data for the 'Gender' column
from sklearn.preprocessing import LabelEncoder
# create an object of the LabelEncoder class
lblEncoder_X = LabelEncoder()
# apply LblEncoder object to our categorical variables (columns - 'Gender') using the fit_transform method. This returns the column encoded.
lblEncoder_X = lblEncoder_X.fit(temp_dataset['Gender']) # we can fit and transform all at once
temp_dataset['Gender'] = lblEncoder_X.transform(temp_dataset['Gender'])

# printing the value counts will show the encoded labels, instead of the original ones
print(temp_dataset['Gender'].value_counts())



Gender
1    253
0     50
2      8
3      1
Name: count, dtype: int64


Now let us try a different encoding strategy, the one-hot encoding. For each feature, this strategy expands the columns in the dataset to have one column per category. A 0/1 indicator can then be used to mark which category corresponds to each data point.

Since we are creating multiple columns from just one, we can not simply replace the content of the original column like we did with the label encoder.

Luckily, sklearn has a helpful function There called `ColumnTransformer` that will take care of things for us. Using `ColumnTransformer` we can specify which transformation we want to apply to each column. We can easily apply different transformations to different columns.

Before using `ColumnTransformer`, let's have a quick look at what OneHotEncoder does.

In [ ]:
from sklearn.preprocessing import OneHotEncoder

# create a 'OneHotEncoder' object
oneHotEncoder = OneHotEncoder()

# create dataset copy for demonstration purposes
temp_dataset = train_dataset_no_nans.copy()

# fit the OneHotEncoder object to feature Gender
onehot_enc = oneHotEncoder.fit(temp_dataset[['Gender']])
print('The categories are: ', onehot_enc.categories_)

# onehot_enc is an array of shape (312,3) containing the one-hot encoding of the feature Gender of the dataframe train_dataset_no_nans
# There is one column for each Gender category.
# Each column has a 1 for each data point that belongs to that category.
# If the correspondence stays the same, it should be:
# the first column is for "Female", second column for "Male", third column for "other"
print(onehot_enc.transform(temp_dataset[['Gender']]).toarray().shape)

# For example, let's see the Gender of the first data point:
print(temp_dataset[['Gender']].head(1))

# Let's see which column is equal to 1.
# If the gender is "Female", the 1 should be in the first column
# If the gender is "Male", the 1 should be in the second column
# If the gender is "other", the 1 should be in the third column
print(onehot_enc.transform(temp_dataset[['Gender']]).toarray()[0])

The categories are:  [array(['Female', 'Male', 'other', nan], dtype=object)]
(312, 4)
   Gender
77   Male
[0. 1. 0. 0.]


Note that if we wanted to continue and add the OneHot encoded columns to the dataset, we would need to do something like this:


```
dum = onehot_enc.transform(temp_dataset[['Gender']]).toarray()           
dum

# we add to the train_dataset_no_nans one feature column called Female and add there the corresponding encoded values
temp_dataset['Gender_Female'] = dum[:,0]     
# we add to the train_dataset_no_nans one feature column called Male and add there the corresponding encoded values
temp_dataset['Gender_Male'] = dum[:,1]  
# add the 'other' category for gender
temp_dataset['Gender_other']= dum[:,2]     

# we delete the (former) column/feature Gender
temp_dataset= temp_dataset.drop(columns='Gender',axis=1)  
```

However, it is not really convenient to repeat the same process for all categorical columns!

Likely, we can use something else.


We can use sklearn function `ColumnTransformer`.
This is a function that applies a given transformation to all the columns of a dataset that we specify. It is a pipeline, that is it can incorporate multiple steps, but here we will use it just for the encoding.

In [ ]:
from sklearn.compose import ColumnTransformer
from sklearn.preprocessing import OneHotEncoder
# define the transformation
# ColumnTransformer takes a list of transformations. Each transformation is expressed
# as a tuple (name, transformer, columns).
ct = ColumnTransformer(
    [
        (
            "onehot_categorical", # --> name of the transformation
            OneHotEncoder(), # --> main function to apply
            ["Gender"], #-->columns to apply it to (we can give more than one column at once!)
        ),
    ],
    remainder="passthrough", #--> what to do with the non-transformed columns. passthrough=keep them
    verbose_feature_names_out=False #--> this keeps columns names simple. Try what happens if you set it as True
)

# create dataset copy for testing purposes
temp_dataset = train_dataset_no_nans.copy()

# the output is an NUMPY ARRAY with the encoded columns.
encoded_array= ct.fit_transform(temp_dataset)

# What if we want a dataframe back? We can combine the array with the info about
# the original and transformed column names.
# This is stored in the ColumnTransformer object, which we called "ct"
# We can access it via the "get_feature_names_out()" attribute like this:
encoded_col_names= ct.get_feature_names_out() #remember python's dot notation
print(encoded_col_names) #note the combined name: original column + category (e.g. Gender_Female)

temp_dataset = pd.DataFrame(encoded_array, columns=encoded_col_names)

print()
print(temp_dataset.head(5))

['Gender_Female' 'Gender_Male' 'Gender_other' 'Gender_nan' 'Loan_ID'
 'Married' 'Dependents' 'Education' 'Self_Employed' 'ApplicantIncome'
 'CoapplicantIncome' 'LoanAmount' 'Loan_Amount_Term' 'Credit_History'
 'Property_Area']

  Gender_Female Gender_Male Gender_other Gender_nan   Loan_ID Married  \
0           0.0         1.0          0.0        0.0  LP002342     Yes   
1           0.0         1.0          0.0        0.0  LP001713     Yes   
2           0.0         1.0          0.0        0.0  LP001194     Yes   
3           0.0         1.0          0.0        0.0  LP001245     Yes   
4           1.0         0.0          0.0        0.0  LP001917      No   

  Dependents     Education Self_Employed ApplicantIncome CoapplicantIncome  \
0          2      Graduate           Yes          1600.0           20000.0   
1          1      Graduate           Yes          7787.0               0.0   
2          2      Graduate            No          2708.0            1167.0   
3          2  Not Gra

##############################################################################






###  Exercise 3

Perform one-hot encoding for ALL the categorical features at once.

Work on `train_dataset_no_nans` directly, without creating a copy. That is, store the final result also in a variable called `train_dataset_no_nans`.

Hints: first, copy the ColumnTransformer object from the cell above and change its name to `ct_cat` (consistently, throughout the code). Then, the goal is to change which columns to apply it to: where do we specify which columns it needs to be applied to? Also remember that we have collected all the categorical features into the variable `categorical_columns`.

In [ ]:
# place your code here



I should note though that this procedure does not necessarily maintain the correct datatypes. To be confident in our data, we can reset them as appropriate. For example, we may want to make sure that all the numerical columns are of dtype float.


In [ ]:
# make sure numerical columns are of type float
train_dataset_no_nans= train_dataset_no_nans.astype(dtype={col: "float64" for col in numerical_columns})


## Label encoding

Now, we also need to encode the labels! You can find the training labels in the **dataframe** called train_labels. For now, they are expressed as Yes/No, but we need numbers.

Remember that the standard procedure for labels is to transform them into integer numbers between 0 and number_of_labels-1. The same way that we applied label encoding to the Gender column a few cells above.

We will store the final results in a numpy array called `train_encoded_labels`.



In [ ]:
# First check: what are the target categories?
print(train_labels.value_counts())

Target
Y         220
N          92
Name: count, dtype: int64


In [ ]:
# create an object of the LabelEncoder class
lblEncoder_Y = LabelEncoder()
# apply LblEncoder object to our target variables
train_encoded_labels = lblEncoder_Y.fit_transform(train_labels['Target'])
print(train_encoded_labels)

[0 1 1 1 1 1 1 1 0 1 0 1 1 1 1 1 1 1 1 1 1 0 1 1 1 1 1 1 1 0 1 1 0 1 1 1 1
 1 1 1 1 1 1 0 1 0 1 1 0 0 1 0 0 1 0 1 1 1 1 0 1 0 0 1 1 1 1 1 1 1 0 1 0 1
 0 1 1 1 1 1 0 1 1 0 1 1 1 1 1 0 1 1 0 1 0 0 0 1 1 1 0 1 0 1 1 0 1 1 1 1 1
 1 1 1 0 1 0 0 0 0 1 0 1 1 1 1 1 0 0 1 1 1 1 1 1 1 0 1 1 1 1 1 1 0 1 1 0 1
 1 1 1 0 0 0 1 1 1 0 0 1 0 1 0 1 0 0 0 1 0 1 0 0 0 1 0 0 0 1 1 1 1 1 0 1 1
 1 1 1 1 1 0 0 1 1 1 1 0 1 0 1 1 1 1 1 1 1 1 1 0 1 1 0 0 0 1 1 1 1 1 1 0 1
 1 1 1 1 1 0 1 1 0 1 1 1 1 1 0 1 1 1 1 1 0 1 0 1 1 1 0 1 1 1 0 1 0 1 1 0 1
 0 1 1 1 1 1 0 1 1 0 1 1 1 0 1 0 1 0 1 1 0 1 0 1 0 1 0 0 1 1 1 1 1 0 1 1 1
 1 1 0 1 1 1 0 1 1 1 0 1 0 1 1 1]


##############################################################################





##5.  Compute Statistics and Check Imbalance

In [ ]:
# Check whether the problem is unbalance
# That is, check whether some target labels are (much) more common than others
# training dataset
sampleClassBias = train_labels['Target'].value_counts()
print('Training Labels distribution:')
print(sampleClassBias)


Training Labels distribution:
Target
Y    220
N     92
Name: count, dtype: int64


Yes - the dataset is unbalanced! There are more labels of type Y than of type N. It's not a massive unbalance, but there are still roughly twice as much "Y" values than there are "N" ones.

We're not going to use any over-sampling or down-sampling today (lots to go through already, so we'll get back to this in a future tutorial). However, we will see how a simple change to one of the arguments of a scikit-learn ML model can already help us take the imbalance into account.


If you're particularly keen, at the end of the tutorial you could have a look at `sklearn.utils.resample` or at one of sklearn contributor package [imbalanced learn](https://github.com/scikit-learn-contrib/imbalanced-learn).

Finally, note that in this case the label is a categorical value that takes 2 values either no (0) or yes (1). It is a **binary classification problem**.

##6.  Feature Scaling

When the data is comprised of feature values with varying scales, many machine learning algorithms can benefit from rescaling the attributes to all have the same scale.


### Implementing Feature Scaling

Feature scaling is a method used to scale the range of variables/values of features.

>![feature scaling](https://drive.google.com/uc?id=1loaMbVo_7ZJo53Ogv7I0XMG_wSOOYBxI)

>  Figure 5: Feature Scaling methods, where x is the observation feature.

There are several ways of scaling the data. One way is called **Standardisation**. During standardisation, for each feature (= each column) we subtract the mean value and divide by the standard deviation. The same transformation is applied to each value within a column.

The other common type of scaling is **normalisation**. In normalisation, for each feature we subtract the minimum value and then divide by the difference between the max and the min of the feature values (that is, the range). Note that dividing by the minimum, means that the new minimum is now 0; dividing by the range means that the new maximum is now 1. So, the new values are all between 0 and 1.

**TL;DR: after scaling, all the variables are now in the same range / same scale, so that no variable is dominated by another.**



In [ ]:
#normalisation - let's work on a copy of the dataset for this one
temp_dataset = train_dataset_no_nans.copy()


# Importing MinMaxScaler and initializing it
from sklearn.preprocessing import MinMaxScaler
min_max_num=MinMaxScaler()
# Normalising (or MinMax scaling) - let's apply to one column only for now
#NOTE THE DOUBLE SQUARE BRACKETS on the right hand side!
# Scalers expect a 2-dimensional input: a dataframe is 2-dimensional, a pandas Series is NOT.
# passing one column in double square brackets tells pandas to return a dataframe rather than a series
# try comparing type(ddummy[['ApplicantIncome']]) and type(ddummy['ApplicantIncome'])
temp_dataset['ApplicantIncome']=min_max_num.fit_transform(temp_dataset[['ApplicantIncome']])

print(temp_dataset[['ApplicantIncome']].head(5))


     ApplicantIncome
77          0.011885
26          0.088881
41          0.025674
203         0.015307
278         0.014511



Question: Do we need to apply feature scaling to the labels? No, it is not needed.

Typically, it is also not needed to scale encoded categorical variables, especially if we OneHot encoded them.

In [ ]:
# Standardisation - let's work on a copy again
temp_dataset = train_dataset_no_nans.copy()

#Standardizing (removing mean and dividing by standard deviation)
from sklearn.preprocessing import StandardScaler
scaler_num= StandardScaler()
temp_dataset['ApplicantIncome']=scaler_num.fit_transform(temp_dataset[['ApplicantIncome']])

print(temp_dataset[['ApplicantIncome']].head(5))



     ApplicantIncome
77         -0.637530
26          0.454698
41         -0.441929
203        -0.588983
278        -0.600281


##############################################################################





###  Exercise 4

Perform standardization OR normalization on all the numerical features at once (hint: you need to replace where we used the specific column ApplicantIncome. Remeber we have a variable that contains all the numerical columns).

You can work directly on `train_dataset_no_nans` now.

Call the scaler of your choice `my_scaler_num`.


In [ ]:
# place code here


##############################################################################






## Quick questions
a) How many columns do we have now in the dataset? How many were there originally?

b) How can we check the results of the scaling procedure?

In [ ]:
# a) How many columns do we have now in the dataset? How many were there originally?
# your code here
print('Before: ', len(train_dataset.columns))
print('After: ', len(train_dataset_no_nans.columns))

Before:  12
After:  12


In [ ]:
# b) How can we check the results of the scaling procedure?
# We can .describe() again: compare with what we got at the beginning.
print(train_dataset_no_nans.describe()) # describe should give us the maximum, minimum and mean values

       ApplicantIncome  CoapplicantIncome  LoanAmount  Loan_Amount_Term  \
count       312.000000         312.000000  312.000000        312.000000   
mean       5211.333333        1555.505243  141.467742        343.755627   
std        5673.665924        2745.719320   75.173565         62.598743   
min         645.000000           0.000000    9.000000         60.000000   
25%        2925.250000           0.000000  101.500000        360.000000   
50%        3901.000000        1083.000000  126.000000        360.000000   
75%        5815.750000        2230.500000  161.250000        360.000000   
max       81000.000000       33837.000000  600.000000        480.000000   

       Credit_History  
count      312.000000  
mean         0.858521  
std          0.348515  
min          0.000000  
25%          1.000000  
50%          1.000000  
75%          1.000000  
max          1.000000  


###  Exercise 5

Perform all the previous data pre-processing steps to the test data and labels.
In order, we did:
1. fill numerical missing values with SimpleImputer (`imptr_num`)
2. fill categorical missing values with SimpleImputer (`imptr_cat`)
3. encoded all categorical values with ColumnTransformer (`ct_cat`)
4. scaled all numerical features with MinMax or StandardScaler (`my_scaler_num`)

Also, we encoded the labels with LabelEncoder (`lblEncoder_Y`)

When transforming the test dataset (both input and labels) we should use transformer objects that have been **fit on the training data**. This is because in real-world applications the test data would not be available during training.

So, we need to call the `.transform` function alone on a previously trained object when it is applied to the test dataset.

Work on the variable `test_dataset_no_nans` which I create on the first line.


In [ ]:
test_dataset_no_nans= test_dataset.copy() #I give you this line of code for free

# Your code here (work on test_dataset_no_nans)


# Logistic regression

Now we are finally ready to apply logistic regression.

In `scikit-learn` we can apply Logistic Regression using either the class [LogisticRegression](https://scikit-learn.org/1.2/modules/generated/sklearn.linear_model.LogisticRegression.html#sklearn.linear_model.LogisticRegression) or [SGDClassifier](https://scikit-learn.org/stable/modules/generated/sklearn.linear_model.SGDClassifier.html#), which is the equivalent of SGDRegression but for classification tasks.

There are some differences between the two, which mostly refer to the exact training algorithm used. Remember that we have seen Gradient Descent as an example, but there are other iterative training algorithm that we can use to train ML models. They all work by training to update parameters little by little to optimise a given loss function and get as high performance as possible.

Also, to be precise, `SGDClassifier` implements multiple linear classification models at once, not only logistic regression, and solves them using a training algorithm based on Gradient Descent. To implement logistic regression specificallt, you need to use it like this, with the argument `loss='log_loss'`: `SGDClassifier(loss='log_loss')`.

(Remember, log-loss is the loss function for binary logistic regression)

Today, **we will use the LogisticRegression class**.


We will check how well the model performs using the accuracy as our performance metric. Remember that accuracy measures what proportion of labels the model has correctly predicted.

In [ ]:
from sklearn.linear_model import SGDClassifier, LogisticRegression
from sklearn.metrics import accuracy_score


# create the Logistic Regression model
log_reg= LogisticRegression(max_iter=1, warm_start=True)

# Remember that .fit() is where we apply the training algorithm! (Gradient Descent or some other one)
# .fit()'s role is to find the parameters for the logistic regression model
# that gives us the best output on the inputs from the training dataset
# It is iterative, so it will update the parameters little by little, each time
# trying to improve the performance a bit

# Let's try and run it first for one iteration only (which is why we set max_iter=1 initially).
# To train, we give what we have already, that is
# the input: train_dataset_no_nans.drop(columns=['Loan_ID']
# and the desired output: train_encoded_labels

# The training algorithm's job is to find the parameters of the ML algorithm
# The ML algorithm's job is to transform the input into the (correct) output using
# those parameters and specific mathematical calculations
# Let's try running just 1 ITERATION of the training algorithm for now.
# We can do this using .partial_fit() instead of .fit()

log_reg.fit(train_dataset_no_nans.drop(columns=['Loan_ID']), train_encoded_labels)
# NOTE: we have dropped the column 'Loan_ID'
# this contains a unique identifier for each data point but it's useless for ML purposes.

# Let's check the parameters: these are our initial guess, so they're probably not very good yet!
parameters0 = log_reg.coef_
intercept0 = log_reg.intercept_
print('Initial parameters theta (there should be one per column, not including Loan_ID).')
print(parameters0)

print('Initial intercept (the b parameter on the lecture slides):')
print(intercept0)

# Let's see how the performance is for now.
# To see the performance, we use the accuracy as our metric.
# The accuracy is the default metric, so can be obtained directly from the ML model object
# using the function .score(), without having to get the predicted labels.
# Let's see the results on both the training and test dataset.

print()
# Checking the model's accuracy (performance)
performance_train0 = log_reg.score(train_dataset_no_nans.drop(columns=['Loan_ID']), train_encoded_labels)
performance_test0 = log_reg.score(test_dataset_no_nans.drop(columns=['Loan_ID']), test_encoded_labels)

print(f'Initial performance on training dataset is {performance_train0:.3f}')
print(f'Initial performance on test dataset is {performance_test0:.3f}')

print('----')
# Now, let's do the full training. That is, we update the parameters for 10000 iterations
log_reg.max_iter = 1000 # let's set the desired higher number of iterations
log_reg.fit(train_dataset_no_nans.drop(columns=['Loan_ID']), train_encoded_labels)

# Let's check the updated parameters
parameters1 = log_reg.coef_
print('Trained parameters')
print(parameters1)

intercept1 = log_reg.intercept_
print(intercept1)

# Let's check the updated performance
performance_train1 = log_reg.score(train_dataset_no_nans.drop(columns=['Loan_ID']), train_encoded_labels)
performance_test1 = log_reg.score(test_dataset_no_nans.drop(columns=['Loan_ID']), test_encoded_labels)

print(f'After training, performance on training dataset is {performance_train1:.3f}')
print(f'After training, performance on test dataset is {performance_test1:.3f}')

# Remember that the real test of generalization should be performed on the test set ONLY!


ValueError: could not convert string to float: 'Male'

Note: if you see a "ConvergenceWarning" it's only because we are running `.fit()` first using only one iteration, which is not enough for the training algorithm to reach the best parameters (that is, to "converge").

Note how both the values for the parameters and the accuracy have changed after the training!

Did performance improve or did it get worse?

In [ ]:
#Let's also get the predictions:
predicted_labels_test = log_reg.predict(test_dataset_no_nans.drop(columns=['Loan_ID']))

# How many of each class have we predicted?
# To do this, we can leverage value_counts after transforming the numpy array into a pandas object
print('Class proportion of predicted labels:')
print(pd.Series(predicted_labels_test).value_counts())

# How many are there of each class in the "correct" labels?
print('Class proportion of correct labels:')
print(pd.Series(test_encoded_labels).value_counts())


It's likely that the minority class is harder to predict.

### Handling class imbalance using `class_weight='balanced'`

One way to try and consider class imbalance when training ML models is to act directly on the loss function. Since the loss function sets the goal of the training algorithm, using a loss function that accounts for class imbalance, in turn changes the final results that we get out of the training algorithm. That is, it changes the final parameters in the hope that we find optimal values that are better for the imbalanced dataset.

We can do so, by using the argument `class_weight='balanced'` with most scikit-learn ML model. Specifically, the “balanced” mode uses the values of the labels to automatically adjust the loss function in a way that gives more importance to the errors that the model makes on the minority class.

Specifically, it weighs each class' contribution to the loss function by a weight that is inversely proportional to class frequencies in the training data (so, the fewer data points a class has, the more weight it adds to the loss function).

Let's try it below.


In [ ]:
# create the Logistic Regression model
log_reg= LogisticRegression(class_weight= 'balanced')

# let's train the model
log_reg.fit(train_dataset_no_nans.drop(columns=['Loan_ID']), train_encoded_labels)

# Let's get the performance (accuracy)
performance_train1 = log_reg.score(train_dataset_no_nans.drop(columns=['Loan_ID']), train_encoded_labels)
performance_test1 = log_reg.score(test_dataset_no_nans.drop(columns=['Loan_ID']), test_encoded_labels)

print(f'After training with balanced weights, performance on training dataset is {performance_train1:.3f}')
print(f'After training with balanced weights, performance on test dataset is {performance_test1:.3f}')

#Let's also get the predictions:
predicted_labels_test = log_reg.predict(test_dataset_no_nans.drop(columns=['Loan_ID']))

# How many of each class have we predicted?
# To do this, we can leverage value_counts after transforming the numpy array into a pandas object
print('Class proportion of predicted labels with balanced weights:')
print(pd.Series(predicted_labels_test).value_counts())

# How many are there of each class in the "correct" labels?
print('Class proportion of correct labels:')
print(pd.Series(test_encoded_labels).value_counts())


Is the final accuracy better or worse? Has anything changed in how many labels we predict for the minority class?